# 02 — UMAP & Clustering

Builds the neighbor graph, computes UMAP, runs Leiden clustering, and visualizes cell types for both datasets independently.

| Dataset | Cells (post-QC) | Genes | PCs for UMAP |
|---------|-----------------|-------|--------------|
| Bhaduri et al. 2020 | ~241,777 | 16,774 | 30 |
| Zhong et al. 2018 | ~2,330 | 20,128 | 20 |

**Prerequisites:** `colab_01_preprocessing.ipynb` must have been run. Both preprocessed h5ad files must be on Drive.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scanpy leidenalg

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

PATHS = {
    'bhaduri_processed': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_preprocessed.h5ad'),
    'zhong_processed':   os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_preprocessed.h5ad'),
    'bhaduri_clustered': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad'),
    'zhong_clustered':   os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_clustered.h5ad'),
}

print('Paths configured.')
for k, v in PATHS.items():
    print(f'  {k}: {v}')

## 2. Load Preprocessed Data

In [ ]:
adata_organoid = sc.read_h5ad(PATHS['bhaduri_processed'])
adata_fetal    = sc.read_h5ad(PATHS['zhong_processed'])

print(f'Bhaduri 2020: {adata_organoid.shape[0]:,} cells x {adata_organoid.shape[1]:,} genes')
print(f'Zhong 2018:   {adata_fetal.shape[0]:,} cells x {adata_fetal.shape[1]:,} genes')
print()
print('Bhaduri obs columns:', list(adata_organoid.obs.columns))
print('Zhong obs columns:  ', list(adata_fetal.obs.columns))

## 2b. Free RAM — Replace Dense X with Sparse Raw Counts

The preprocessed h5ad stores a **dense scaled X matrix** (~16 GB for Bhaduri) because `sc.pp.scale()` in colab_01 destroys sparsity. We don't need X for the neighbor graph, UMAP, or Leiden — those only use `adata.obsm['X_pca']`. Replacing X with the sparse raw counts from `adata.raw` frees ~25 GB of RAM.

In [ ]:
# Replace dense scaled X with sparse raw counts to free ~14 GB RAM
adata_organoid.X = adata_organoid.raw.X.copy()

import gc
gc.collect()

print('Dense X replaced with sparse raw counts.')
print(f'adata_organoid.X type: {type(adata_organoid.X)}')

## 3. Bhaduri — Inspect Metadata

Before UMAP we check what metadata is available. Bhaduri's barcodes may encode sample or protocol information — if so, we can color the UMAP by it later to check for batch effects.

In [ ]:
# Look at the first few barcodes — sample/protocol info is often embedded in them
print('First 10 barcodes:')
print(adata_organoid.obs_names[:10].tolist())
print()

# Try to extract sample ID from barcode (common pattern: BARCODE_SAMPLEID or SAMPLEID_BARCODE)
# Adjust the split character and index if needed after inspecting the barcodes above
try:
    adata_organoid.obs['sample'] = adata_organoid.obs_names.str.split('_', n=1).str[0]
    print(f'Extracted sample IDs: {adata_organoid.obs["sample"].nunique()} unique values')
    print(adata_organoid.obs['sample'].value_counts().head(20))
except Exception as e:
    print(f'Could not extract sample info: {e}')

## 4. Bhaduri — Neighbor Graph

Build a k-nearest-neighbor graph in the PCA space. UMAP and Leiden clustering both use this graph — it's the single most important parameter choice after the number of PCs.

- `n_pcs=30` — the number we chose from the elbow plot in colab_01
- `n_neighbors=30` — for a large, complex dataset (242k cells, 3 protocols), more neighbors give a smoother UMAP with better global structure. Default is 15.

**Note:** This will take a few minutes on 242k cells.

In [ ]:
sc.pp.neighbors(adata_organoid, n_neighbors=30, n_pcs=30)
print('Bhaduri: neighbor graph built.')

## 5. Bhaduri — UMAP

UMAP projects the neighbor graph into 2D. Cells that are similar in gene expression land close together.

**Note:** On 242k cells this will take 5–15 minutes.

In [ ]:
sc.tl.umap(adata_organoid)
print('Bhaduri: UMAP computed.')

## 6. Bhaduri — Leiden Clustering

Leiden detects communities in the neighbor graph. `resolution` controls granularity:
- Lower (0.3) → fewer, broader clusters
- Higher (1.0+) → more, finer clusters

Bhaduri has 3 organoid protocols and many cell types — expect 15–30 clusters at resolution 0.5. We'll start there and adjust after inspecting the UMAP.

In [ ]:
LEIDEN_RES_ORGANOID = 0.5  # adjust after looking at the UMAP

sc.tl.leiden(adata_organoid, resolution=LEIDEN_RES_ORGANOID)

n_clusters = adata_organoid.obs['leiden'].nunique()
print(f'Bhaduri: {n_clusters} clusters at resolution={LEIDEN_RES_ORGANOID}')
print(adata_organoid.obs['leiden'].value_counts().sort_index())

## 7. Bhaduri — Visualize UMAP

In [ ]:
# Color by Leiden cluster
sc.pl.umap(adata_organoid, color='leiden', legend_loc='on data', title='Bhaduri 2020 — Leiden clusters')

In [ ]:
# Color by QC metrics — check if any cluster is driven by technical noise
# High MT% cluster = stress/dying cells; high n_genes cluster = remaining doublets
sc.pl.umap(
    adata_organoid,
    color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    title=['Bhaduri — n_genes', 'Bhaduri — total_counts', 'Bhaduri — MT%']
)

In [ ]:
# Color by sample — check for batch effects (if sample info was extracted above)
# If all one sample, skip this
if 'sample' in adata_organoid.obs.columns and adata_organoid.obs['sample'].nunique() > 1:
    sc.pl.umap(adata_organoid, color='sample', title='Bhaduri 2020 — by sample')
else:
    print('No sample metadata available — skipping batch check.')

In [ ]:
# Known brain cell type markers — where do they land on the UMAP?
# This gives a first pass at cell type identity before formal annotation
brain_markers = [
    'SOX2',    # radial glia / neural progenitors
    'PAX6',    # radial glia
    'EOMES',   # intermediate progenitors (TBR2)
    'TBR1',    # early excitatory neurons
    'NEUROD2', # excitatory neurons
    'GAD1',    # inhibitory neurons
    'GAD2',    # inhibitory neurons
    'GFAP',    # astrocytes
    'MKI67',   # cycling cells
]

# Only plot markers that are actually in the gene list
available = [g for g in brain_markers if g in adata_organoid.var_names]
missing   = [g for g in brain_markers if g not in adata_organoid.var_names]
print(f'Available markers: {available}')
print(f'Missing from dataset: {missing}')

if available:
    sc.pl.umap(adata_organoid, color=available, ncols=3, title=[f'Bhaduri — {g}' for g in available])

## 8. Bhaduri — Marker Genes

Find the most differentially expressed genes for each cluster compared to all others. This tells us what cell type each cluster likely is.

We use the Wilcoxon rank-sum test — it's robust and the standard choice for scRNA-seq.

In [ ]:
sc.tl.rank_genes_groups(adata_organoid, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata_organoid, n_genes=10, sharey=False)

In [ ]:
# Dot plot — expression of known markers across clusters
# Dot size = fraction of cells expressing; color = mean expression
if available:
    sc.pl.dotplot(adata_organoid, available, groupby='leiden', standard_scale='var',
                  title='Bhaduri — known brain markers by cluster')

---

## 9. Zhong — Inspect Metadata

Zhong's barcodes may encode gestational week (e.g. GW8, GW10, ...) — if so we can color the UMAP by developmental stage.

In [ ]:
print('First 10 barcodes:')
print(adata_fetal.obs_names[:10].tolist())
print()
print('Obs columns:', list(adata_fetal.obs.columns))

In [ ]:
# Try to extract gestational week from barcode
# Typical Zhong barcodes look like: GW8_cell1, or include GW info as prefix
import re

def extract_gw(barcode):
    match = re.search(r'GW(\d+)', barcode, re.IGNORECASE)
    return f'GW{match.group(1)}' if match else 'unknown'

adata_fetal.obs['gestational_week'] = [extract_gw(b) for b in adata_fetal.obs_names]

gw_counts = adata_fetal.obs['gestational_week'].value_counts().sort_index()
print('Gestational weeks found:')
print(gw_counts)

## 10. Zhong — Neighbor Graph

- `n_pcs=20` — chosen from elbow plot in colab_01 (sharp elbow at ~PC20)
- `n_neighbors=15` — default, appropriate for the smaller dataset (2,330 cells)

In [ ]:
sc.pp.neighbors(adata_fetal, n_neighbors=15, n_pcs=20)
print('Zhong: neighbor graph built.')

## 11. Zhong — UMAP

In [ ]:
sc.tl.umap(adata_fetal)
print('Zhong: UMAP computed.')

## 12. Zhong — Leiden Clustering

In [ ]:
LEIDEN_RES_FETAL = 0.5  # adjust after looking at the UMAP

sc.tl.leiden(adata_fetal, resolution=LEIDEN_RES_FETAL)

n_clusters = adata_fetal.obs['leiden'].nunique()
print(f'Zhong: {n_clusters} clusters at resolution={LEIDEN_RES_FETAL}')
print(adata_fetal.obs['leiden'].value_counts().sort_index())

## 13. Zhong — Visualize UMAP

In [ ]:
sc.pl.umap(adata_fetal, color='leiden', legend_loc='on data', title='Zhong 2018 — Leiden clusters')

In [ ]:
sc.pl.umap(
    adata_fetal,
    color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    title=['Zhong — n_genes', 'Zhong — total_counts', 'Zhong — MT%']
)

In [ ]:
# Color by gestational week — should see temporal ordering in the UMAP if the
# transcriptional program captures developmental progression
if adata_fetal.obs['gestational_week'].nunique() > 1:
    sc.pl.umap(adata_fetal, color='gestational_week', title='Zhong 2018 — gestational week')
else:
    print('Could not extract gestational week — check barcodes.')

In [ ]:
available_fetal = [g for g in brain_markers if g in adata_fetal.var_names]
missing_fetal   = [g for g in brain_markers if g not in adata_fetal.var_names]
print(f'Available markers: {available_fetal}')
print(f'Missing from dataset: {missing_fetal}')

if available_fetal:
    sc.pl.umap(adata_fetal, color=available_fetal, ncols=3, title=[f'Zhong — {g}' for g in available_fetal])

## 14. Zhong — Marker Genes

In [ ]:
sc.tl.rank_genes_groups(adata_fetal, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata_fetal, n_genes=10, sharey=False)

In [ ]:
if available_fetal:
    sc.pl.dotplot(adata_fetal, available_fetal, groupby='leiden', standard_scale='var',
                  title='Zhong — known brain markers by cluster')

## 15. Save Clustered Objects

In [ ]:
adata_organoid.write_h5ad(PATHS['bhaduri_clustered'])
adata_fetal.write_h5ad(PATHS['zhong_clustered'])

for k in ['bhaduri_clustered', 'zhong_clustered']:
    size_mb = os.path.getsize(PATHS[k]) / 1e6
    print(f'Saved: {PATHS[k]} ({size_mb:.1f} MB)')

## Done

Both datasets have been clustered:
- Neighbor graph built (30 PCs / Bhaduri, 20 PCs / Zhong)
- UMAP computed
- Leiden clusters assigned (resolution=0.5 — adjust if needed)
- Marker genes ranked per cluster
- UMAP colored by clusters, QC metrics, and known brain markers

Saved to Drive:
- `data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad`
- `data/processed/zhong_2018/zhong_2018_clustered.h5ad`

**Next steps:**
1. Inspect the UMAPs and adjust Leiden resolution if the cluster granularity is off
2. Annotate clusters with cell type labels using the marker genes
3. `colab_03_integration.ipynb` — Harmony or scVI to align both datasets